# Train Basic Hairstyle Attribute Predictor

Lightweight multi-head classifier for the reviewed kept-asset basic fields: `length` and `curl`.

In [1]:
import json
from pathlib import Path
import sys
import time

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch.utils.data import DataLoader

from systems.static_auto_tryon.auto_app.ml.datasets import MultiAttributeDataset, read_jsonl_manifest
from systems.static_auto_tryon.auto_app.ml.hairstyle_attribute_model import build_attribute_model, multitask_cross_entropy
from systems.static_auto_tryon.auto_app.ml.metrics import attribute_accuracy, exact_match_accuracy
from systems.static_auto_tryon.auto_app.ml.transforms import ResizeImage

PROJECT_ROOT

WindowsPath('.')

In [2]:
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'reviewed_hairstyle_basic'
TRAIN_MANIFEST = DATASET_ROOT / 'train.jsonl'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'reviewed_hairstyle_basic' / 'basic_attribute_model.pt'

REQUIRE_GPU = True
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)

CORE_FIELDS = ('length', 'curl')


In [3]:
train_records = read_jsonl_manifest(TRAIN_MANIFEST)
val_records = read_jsonl_manifest(VAL_MANIFEST)
label_vocab = json.loads(VOCAB_PATH.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

attribute_vocab_sizes = {field: len(label_vocab[field]) for field in CORE_FIELDS}

print('Device:', DEVICE)
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('Core fields:', CORE_FIELDS)
print('Train records:', len(train_records))
print('Val records:', len(val_records))
summary

Device: cuda
Torch version: 2.11.0+cu128
CUDA version: 12.8
Core fields: ('length', 'curl')
Train records: 67
Val records: 17


{'core_fields': ['length', 'curl'],
 'total_records': 84,
 'train_records': 67,
 'val_records': 17,
 'train_ratio': 0.8,
 'field_value_counts': {'length': {'long': 44, 'short': 35, 'medium': 5},
  'curl': {'straight': 45, 'wavy': 37, 'curly': 2}}}

In [4]:
transform = ResizeImage((IMAGE_SIZE, IMAGE_SIZE))

train_dataset = MultiAttributeDataset(
    train_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=CORE_FIELDS,
)
val_dataset = MultiAttributeDataset(
    val_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=CORE_FIELDS,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
batch['image'].shape, {field: tensor.shape for field, tensor in batch['labels'].items()}

(torch.Size([16, 3, 224, 224]),
 {'length': torch.Size([16]), 'curl': torch.Size([16])})

In [5]:
model = build_attribute_model(attribute_vocab_sizes=attribute_vocab_sizes, base_channels=32, dropout=0.2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

sum(parameter.numel() for parameter in model.parameters())

1174758

In [6]:
def move_targets_to_device(targets, device):
    return {field: tensor.to(device) for field, tensor in targets.items()}

def filter_outputs(outputs, fields):
    return {field: outputs[field] for field in fields}

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    total_batches = 0
    for batch in loader:
        images = batch['image'].to(device)
        targets = move_targets_to_device(batch['labels'], device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        outputs = filter_outputs(outputs, CORE_FIELDS)
        loss = multitask_cross_entropy(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())
        total_batches += 1
    return running_loss / max(total_batches, 1)

def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    total_batches = 0
    output_batches = []
    target_batches = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            targets = move_targets_to_device(batch['labels'], device)
            outputs = model(images)
            outputs = filter_outputs(outputs, CORE_FIELDS)
            loss = multitask_cross_entropy(outputs, targets)
            running_loss += float(loss.item())
            total_batches += 1
            output_batches.append({field: tensor.detach().cpu() for field, tensor in outputs.items()})
            target_batches.append({field: tensor.detach().cpu() for field, tensor in targets.items()})

    merged_outputs = {
        field: torch.cat([batch[field] for batch in output_batches], dim=0)
        for field in CORE_FIELDS
    }
    merged_targets = {
        field: torch.cat([batch[field] for batch in target_batches], dim=0)
        for field in CORE_FIELDS
    }
    attr_scores = attribute_accuracy(merged_outputs, merged_targets)
    return {
        'val_loss': running_loss / max(total_batches, 1),
        'exact_match_accuracy': exact_match_accuracy(merged_outputs, merged_targets),
        **attr_scores,
    }


In [7]:
history = []
best_metric = float('-inf')
best_epoch = None
epoch_durations = []

for epoch in range(1, EPOCHS + 1):
    print(f'Running epoch {epoch}/{EPOCHS}...')
    epoch_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    metrics = evaluate(model, val_loader, DEVICE)
    epoch_duration = time.time() - epoch_start
    epoch_durations.append(epoch_duration)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'epoch_seconds': round(epoch_duration, 2),
        **metrics,
    }
    history.append(row)

    current_metric = metrics['exact_match_accuracy']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch

    average_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
    remaining_seconds = average_epoch_seconds * (EPOCHS - epoch)

    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history,
            'last_epoch': epoch,
            'core_fields': CORE_FIELDS,
            'label_vocab': label_vocab,
        },
        CHECKPOINT_PATH,
    )

    print(f"Completed epoch {epoch}/{EPOCHS}")
    print(f"Epoch time: {format_seconds(epoch_duration)}")
    print(f"Estimated remaining: {format_seconds(remaining_seconds)}")
    print(f"Best epoch so far: {best_epoch} (exact_match_accuracy={best_metric:.4f})")
    print(row)
    display(pd.DataFrame(history))

print(f'Saved checkpoint to {CHECKPOINT_PATH}')
pd.DataFrame(history)

Running epoch 1/30...
Completed epoch 1/30
Epoch time: 00:01
Estimated remaining: 00:50
Best epoch so far: 1 (exact_match_accuracy=0.0000)
{'epoch': 1, 'train_loss': 2.0940362930297853, 'epoch_seconds': 1.73, 'val_loss': 2.2906482219696045, 'exact_match_accuracy': 0.0, 'length': 0.0, 'curl': 0.4117647111415863}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.094036,1.73,2.290648,0.0,0.0,0.411765


Running epoch 2/30...
Completed epoch 2/30
Epoch time: 00:00
Estimated remaining: 00:28
Best epoch so far: 2 (exact_match_accuracy=0.2941)
{'epoch': 2, 'train_loss': 1.9759682178497315, 'epoch_seconds': 0.33, 'val_loss': 2.327877640724182, 'exact_match_accuracy': 0.29411765933036804, 'length': 0.47058823704719543, 'curl': 0.4117647111415863}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.094036,1.73,2.290648,0.000000,0.000000,0.411765
1,2,1.975968,0.33,2.327878,0.294118,0.470588,0.411765


Running epoch 3/30...
Completed epoch 3/30
Epoch time: 00:00
Estimated remaining: 00:21
Best epoch so far: 2 (exact_match_accuracy=0.2941)
{'epoch': 3, 'train_loss': 1.906791877746582, 'epoch_seconds': 0.33, 'val_loss': 2.317282199859619, 'exact_match_accuracy': 0.29411765933036804, 'length': 0.47058823704719543, 'curl': 0.4117647111415863}


...

,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.094036,1.73,2.290648,0.000000,0.000000,0.411765
1,2,1.975968,0.33,2.327878,0.294118,0.470588,0.411765
2,3,1.906792,0.33,2.317282,0.294118,0.470588,0.411765
3,4,1.567603,0.34,2.487076,0.294118,0.470588,0.411765
4,5,1.508034,0.30,2.799252,0.294118,0.470588,0.411765
5,6,1.432619,0.28,2.815501,0.294118,0.470588,0.411765
6,7,1.322989,0.32,2.457397,0.294118,0.470588,0.411765
7,8,1.064182,0.30,2.077427,0.294118,0.470588,0.352941
8,9,1.129021,0.29,1.614919,0.294118,0.470588,0.411765
9,10,1.026514,0.29,1.727277,0.411765,0.529412,0.470588


Running epoch 30/30...
Completed epoch 30/30
Epoch time: 00:00
Estimated remaining: 00:00
Best epoch so far: 14 (exact_match_accuracy=0.5294)
{'epoch': 30, 'train_loss': 0.4604868531227112, 'epoch_seconds': 0.32, 'val_loss': 0.9454998597502708, 'exact_match_accuracy': 0.47058823704719543, 'length': 0.7058823704719543, 'curl': 0.5882353186607361}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.094036,1.73,2.290648,0.000000,0.000000,0.411765
1,2,1.975968,0.33,2.327878,0.294118,0.470588,0.411765
2,3,1.906792,0.33,2.317282,0.294118,0.470588,0.411765
3,4,1.567603,0.34,2.487076,0.294118,0.470588,0.411765
4,5,1.508034,0.30,2.799252,0.294118,0.470588,0.411765
5,6,1.432619,0.28,2.815501,0.294118,0.470588,0.411765
6,7,1.322989,0.32,2.457397,0.294118,0.470588,0.411765
7,8,1.064182,0.30,2.077427,0.294118,0.470588,0.352941
8,9,1.129021,0.29,1.614919,0.294118,0.470588,0.411765
9,10,1.026514,0.29,1.727277,0.411765,0.529412,0.470588


Saved checkpoint to backend/models\reviewed_hairstyle_basic\basic_attribute_model.pt


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.094036,1.73,2.290648,0.000000,0.000000,0.411765
1,2,1.975968,0.33,2.327878,0.294118,0.470588,0.411765
2,3,1.906792,0.33,2.317282,0.294118,0.470588,0.411765
3,4,1.567603,0.34,2.487076,0.294118,0.470588,0.411765
4,5,1.508034,0.30,2.799252,0.294118,0.470588,0.411765
5,6,1.432619,0.28,2.815501,0.294118,0.470588,0.411765
6,7,1.322989,0.32,2.457397,0.294118,0.470588,0.411765
7,8,1.064182,0.30,2.077427,0.294118,0.470588,0.352941
8,9,1.129021,0.29,1.614919,0.294118,0.470588,0.411765
9,10,1.026514,0.29,1.727277,0.411765,0.529412,0.470588
